# Four-model B1 logistic stack

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

The stack was explored after inspecting other dev results, so comparisons across stack families on the same official dev set are post-selection and may be optimistic. The train-calibration CV is not an unseen test estimate. Reproduction needs the saved per-model probability arrays and selected checkpoint lineage in the original local B1 archive; the small package includes the builder script and full hash/provenance record, but not all probabilities or model weights.

![Preserved computation graph](../figures/architecture.svg)

## Task and architecture

This is a prediction-level ensemble, not a newly trained Phase 1/B1 backbone. It joins saved calibration, dev and test probabilities from: v7_transformer, v6_transformer, v5_gru, v4_reference. An L2-regularized logistic regression learns the mixing weights from 512 training-calibration examples; each original model retains its own checkpoint. Decision threshold 0.500000.

**Adaptation:** Four inputs fixed from prior evidence; L2 logistic stack C=0.1 fit on 512 disjoint train calibration examples; five-fold calibration CV; threshold 0.5. Official dev used only to report.

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Exact mechanism and experimental controls

### Provenance and scientific boundary

[CODE] Prediction-level B1 ensemble, not an additional neural backbone.

[CODE] For a case, stack the base-model probabilities into vector p. Compute logit = intercept + Σ(coefficient_i × p_i), apply sigmoid, and output 1 if the resulting probability reaches the recorded threshold. Every base probability comes from an independently trained B1 model; no backbone weight is updated when fitting the stack.

[CODE/RUN] L2-regularized logistic regression combines 4 aligned model probabilities: v7_transformer, v6_transformer, v5_gru, v4_reference. The stored coefficients are 0.5568, 0.3330, 0.3002, 0.3513 and intercept -0.0340.

### Exact recorded configuration

| Setting | Recorded value |
|---|---|
| Input models | `v7_transformer, v6_transformer, v5_gru, v4_reference` |
| Calibration rows | `512` |
| Regularizer | `L2 logistic regression` |
| Decision threshold | `0.5` |
| 5-fold calibration CV macro-F1 | `0.7106549364613881` |
| Official dev macro-F1 | `0.6939555987684329` |
| Test predictions | `1517` |
| Output ZIP SHA-256 | `b341bb9d03656077c32ffd77cf4c60f9cb38ac55f6d7e583cb40080d0761e85e` |

### Optimization and selection

[CODE] There are no optimizer epochs for a new backbone here. The logistic combiner fits probability columns with L2 regularization. The stack with CV-calibrated threshold chooses its cutoff using out-of-fold calibration scores; the other stacks use 0.5.

### Evaluation protocol

[RUN] The stack was fitted only on 512 train-calibration rows, and five-fold calibration CV macro-F1 was 0.710655. Official dev macro-F1 0.693956 on 994 cases is exploratory because multiple stack families were compared after seeing dev. The hidden Codabench score is a separate user-supplied observation; calibration CV is not an unseen test result.

### Reproduction lineage

The original builder script and verified input-array hashes are in code/ and evidence/. Original base-model probability arrays and checkpoint backups remain in the local B1 study archive.

## Recorded result

Five-fold train-calibration CV macro-F1 **0.710655** on 512 cases. Exploratory official 994-case dev macro-F1 **0.693956**. Verified 1,517-ID ZIP SHA-256 `b341bb9d03656077c32ffd77cf4c60f9cb38ac55f6d7e583cb40080d0761e85e`.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Preserved run trace

![Recorded training loss or fitted stack coefficients](../figures/recorded_trace.svg)

The figure is drawn from copied run records, not from an execution of this companion notebook. The machine-readable source is in `evidence/`.

## Source excerpt: `build_b1_stack_submission.py`

The complete source is in `code/`. This excerpt illustrates the preserved implementation; it is not a training replay.

In [ ]:
import datetime
import hashlib
import json
import pathlib
import sys
import zipfile

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = pathlib.Path(sys.argv[1])
ready = pathlib.Path(sys.argv[2])
outputs = root / 'outputs'
models = {
    'v7_transformer': ('v7', 'B1_Window_TransformerOnly'),
    'v6_transformer': ('v6', 'B1_Window_TransformerOnly'),
    'v5_gru': ('v5', 'B1_Window_GRUOnly'),
    'v4_reference': ('v4', 'B1_FullData_Reference'),
}
assert (root / 'VERIFIED_HEADS_V7_SEGMENT3_BACKUPS.json').exists()
assert (root / 'VERIFIED_HEADS_V7_SEGMENT3_EXPORTS.json').exists()


def sha256(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def load_model(version, name):
    base = outputs / version / 'CS781_B1_Hierarchy_Depth' / 'runs' / name / 'exports'
    dirs = [p for p in base.iterdir() if p.is_dir() and p.name.startswith('segment_')]
    exp = max(dirs, key=lambda p: int(p.name.split('_')[-1]))
    data = {}
    hashes = {}
    for split in ('calibration', 'dev', 'test'):
        path = exp / f'{split}_probabilities.npz'
        hashes[split] = {'path': str(path), 'sha256': sha256(path)}
        with np.load(path, allow_pickle=False) as arr:
            ids = arr['ids'].tolist()
            logits = arr['logits']
            assert logits.shape == (len(ids), 2)
            assert len(ids) == len(set(ids))
            margin = (logits[:, 1] - logits[:, 0]).astype(np.float64)
            assert np.isfinite(margin).all()
            entry = {'ids': ids, 'margin': margin}
            if split == 'calibration':
                entry['labels'] = arr['labels'].astype(int)
            elif split == 'dev':
                rows = json.loads((exp / 'dev_truth.json').read_text(encoding='utf-8'))
                gold = {r['id']: int(r['gold']) for r in rows}
                assert len(gold) == len(ids)
                entry['labels'] = np.array([gold[i] for i in ids], dtype=int)
            data[split] = entry
    return exp, data, hashes

loaded = {key: load_model(*spec) for key, spec in models.items()}
order = list(models)
base = loaded[order[0]][1]
with zipfile.ZipFile(ready / 'B1_Transformer_v7.zip') as archive:
    official_ids = [r['id'] for r in json.loads(archive.read('prediction.json'))]
assert len(official_ids) == len(set(official_ids)) == 1517


def matrix(split):
    ids = base[split]['ids']
    assert len(ids) == (1517 if split == 'test' else 512 if split == 'calibration' else 994)
    columns = []
    labels = base[split].get('labels')
    for key in order:
        entry = loaded[key][1][split]
        positions = {identifier: i for i, identifier in enumerate(entry['ids'])}
        assert set(positions) == set(ids)
        idx = [positions[identifier] for identifier in ids]
        columns.append(entry['margin'][idx])
        if labels is not None:
            assert np.array_equal(entry['labels'][idx], labels)
    return ids, np.column_stack(columns), labels

cal_ids, x_cal, y_cal = matrix('calibration')
dev_ids, x_dev, y_dev = matrix('dev')
test_ids, x_test, _ = matrix('test')
assert set(test_ids) == set(official_ids)
assert np.unique(y_cal, return_counts=True)[1].tolist() == [256, 256]
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000, random_state=0))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=781)
cv_probability = cross_val_predict(model, x_cal, y_cal, cv=folds, method='predict_proba')[:, 1]
cv_f1 = float(f1_score(y_cal, cv_probability >= 0.5, average='macro'))
model.fit(x_cal, y_cal)
dev_probability = model.predict_proba(x_dev)[:, 1]
dev_f1 = float(f1_score(y_dev, dev_probability >= 0.5, average='macro'))
test_probability = model.predict_proba(x_test)[:, 1]
lookup = {identifier: int(prob >= 0.5) for identifier, prob in zip(test_ids, test_probability)}
rows = [{'id': identifier, 'pred': lookup[identifier]} for identifier in official_ids]
assert len(rows) == 1517 and all(type(row['pred']) is int and row['pred'] in (0, 1) for row in rows)

name = 'B1_Stack_v7v6v5v4.zip'
assert len(name) <= 64
zip_path = ready / name
if zip_path.exists():
    raise RuntimeError('Refusing to overwrite existing submission ZIP')
encoded = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('prediction.json', encoded)
with zipfile.ZipFile(zip_path) as archive:
    assert archive.namelist() == ['prediction.json'] and archive.testzip() is None
    restored = json.loads(archive.read('prediction.json'))

## Verify the copied run evidence

The next cell is a read-only inspection helper. It does not train or generate new scores. Running it is optional; this delivered notebook has no saved execution output.

In [ ]:
from pathlib import Path
import json
root = Path('..')
for record in sorted((root / 'evidence').glob('*terminal.json')):
    result = json.loads(record.read_text(encoding='utf-8'))
    print(record.name, result.get('name'), result.get('state'), result.get('dev_metrics', {}).get('macro_f1'))


## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun. The source code can be reviewed without executing it.